In [1]:
import pyspark.sql.functions as f
import pyspark.ml as ml
import seaborn as sns
from pyspark.sql.types import *
import matplotlib.pyplot as plt

In [2]:
data = spark.read.parquet("./data_prime_auto_clean.parquet")

In [3]:
feature_cols=[
    'annee_naissance',
     'annee_permis',
     'puis_fiscale',
     'anc_veh',
     'kmage_annuel',
     'crm',
     'energie_veh_ohe',
    
]

cat_cols = ["energie_veh"]

ml_stages = []

for cc in cat_cols:
    indexer = ml.feature.StringIndexer(inputCol=cc,outputCol=cc+"_i")
    ml_stages.append(indexer)

for cc in cat_cols:
    indexer = ml.feature.OneHotEncoder(inputCol=cc+"_i",outputCol=cc+"_ohe")
    ml_stages.append(indexer)

va = ml.feature.VectorAssembler().setInputCols(feature_cols).setOutputCol("features")

ml_stages.append(va)


In [4]:
lr_trainer = ml.regression.LinearRegression(featuresCol="features",labelCol="prime_tot_ttc",predictionCol="predictions") 
rf_trainer = ml.regression.RandomForestRegressor(featuresCol="features",labelCol="prime_tot_ttc",predictionCol="predictions") 

lr_pipeline = ml.Pipeline(stages = ml_stages + [lr_trainer])
rf_pipeline = ml.Pipeline(stages = ml_stages + [rf_trainer])


In [5]:
train,test = data.randomSplit([0.6,0.4],1)

In [6]:
lr_model = lr_pipeline.fit(train)

24/03/20 14:12:54 WARN Instrumentation: [9ad2c60f] regParam is zero, which might cause numerical instability and overfitting.
24/03/20 14:12:55 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/03/20 14:12:57 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [7]:
rf_model = rf_pipeline.fit(train)

In [8]:
def evaluate_model(model,train,test):
    metrics = {"mae","rmse","mse"}
    results={}
    evaluator = ml.evaluation.RegressionEvaluator(predictionCol="predictions",labelCol="prime_tot_ttc")
    for metric in metrics:
        results[metric+"_train"]=evaluator.setMetricName(metric).evaluate(model.transform(train))
        results[metric+"_test"]=evaluator.setMetricName(metric).evaluate(model.transform(test))
    return results


In [9]:
evaluate_model(lr_model,train,test)

{'mae_train': 59.60856826947569,
 'mae_test': 59.572334880457774,
 'rmse_train': 78.61875904889995,
 'rmse_test': 78.82138291741029,
 'mse_train': 6180.909274388987,
 'mse_test': 6212.810405013019}

In [10]:
evaluate_model(rf_model,train,test)

{'mae_train': 48.762255310225186,
 'mae_test': 48.769628471873276,
 'rmse_train': 65.85657204100207,
 'rmse_test': 66.11371703098334,
 'mse_train': 4337.088080991695,
 'mse_test': 4371.023579652937}

In [11]:
pgb = ml.tuning.ParamGridBuilder().addGrid(rf_trainer.maxDepth,[5,8]).addGrid(rf_trainer.numTrees,[20,25]).build()

In [12]:
mae_evaluator = ml.evaluation.RegressionEvaluator(predictionCol="predictions",labelCol="prime_tot_ttc",metricName="mae")

In [17]:
gs_trainer = ml.tuning.CrossValidator(estimator = rf_pipeline,evaluator = mae_evaluator, estimatorParamMaps=pgb,numFolds=3,collectSubModels=True)

In [26]:
gs_model = gs_trainer.fit(data)

24/03/20 14:35:16 WARN DAGScheduler: Broadcasting large task binary with size 1091.9 KiB
24/03/20 14:35:40 WARN DAGScheduler: Broadcasting large task binary with size 1093.9 KiB
24/03/20 14:36:05 WARN DAGScheduler: Broadcasting large task binary with size 1094.5 KiB


In [25]:
gs_model.avgMetrics

[48.424500740355775, 49.01794740052961, 43.614420046970714, 43.68339463049558]

In [23]:
gs_model.collectSubModels

AttributeError: 'CrossValidatorModel' object has no attribute 'collectSubModels'

In [29]:
import os
from random import random, randint
import mlflow
from mlflow import log_metric, log_param, log_artifacts

mlflow.set_tracking_uri(uri="file:///opt/workspace/mlflow")

experiment_name = "mlflow_training"
mlflow.set_experiment(experiment_name) 

with mlflow.start_run():
    log_param("parametre", randint(0, 100))
    
    log_metric("mae", random(),1)
    log_metric("mae", random() ,2 )
    log_metric("mae", random() ,3)
    
    if not os.path.exists("outputs"):
        os.makedirs("outputs")
    with open("outputs/model.txt", "w") as f:
        f.write("exemple d'artéfact")
    
    log_artifacts("outputs")


In [30]:
from  mlflow.spark import log_model
import mlflow
from mlflow import log_metric, log_param, log_artifacts
experiment_name = "mlflow_training"
mlflow.set_tracking_uri(uri="file:///opt/workspace/mlflow")
mlflow.set_experiment(experiment_name) 
with mlflow.start_run():
    mlflow.spark.log_model(rf_model,artifact_path="rf_model",registered_model_name="rf_model_1")
    evaluation = evaluate_model(rf_model,train,test)
    mlflow.log_metric("mae",evaluation["mae_test"],1)

Successfully registered model 'rf_model_1'.
Created version '1' of model 'rf_model_1'.
